In [2]:
#案例：代码实现RNN 全球人名分类案例，录入人民，预测其国家

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F                     # 常用的函数库...
import torch.optim as optim                         # 优化器模块
from  torch.utils.data import Dataset, DataLoader   # 数据集对象, 数据加载器
import string                                       # 字符串处理模块.
import time                                         # 时间模块.
import matplotlib.pyplot as plt        

D:\Anaconda\envs\stock1\lib\site-packages\torch\cuda\__init__.py:83: UserWarning: CUDA initialization: CUDA driver initialization failed, you might not have a CUDA gpu. (Triggered internally at  ..\c10\cuda\CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


In [4]:
# 解决绘图时, 中文乱码问题.
plt.rcParams['font.sans-serif'] = ['SimHei']        # Mac本换成: 'Arial Unicode  MS'
plt.rcParams['axes.unicode_minus'] = False

In [6]:
# todo1. 定义遍历，获取常用的字符数量
#1、获取所有的常用字符
all_letters = string.ascii_letters + " .,,'"
#2、获取常用的字符的数量
n_letters = len(all_letters)
print('所有常用字符: ', all_letters)          # abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ .,;'
print('常用字符数量: ', n_letters)            # 57


所有常用字符:  abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ .,,'
常用字符数量:  57


In [7]:
#2、 定义遍历，获取常用国家名，种类数和个数
#1 国家名，种类数
categories = ['Italian', 'English', 'Arabic', 'Spanish', 'Scottish', 'Irish', 'Chinese', 'Vietnamese', 'Japanese', 'French', 'Greek', 'Dutch', 'Korean', 'Polish', 'Portuguese', 'Russian', 'Czech', 'German']
# 2. 国家名 个数.
category_num = len(categories)
print('国家名: ', categories)
print('国家名种类数: ', category_num)         # 18个国家名

国家名:  ['Italian', 'English', 'Arabic', 'Spanish', 'Scottish', 'Irish', 'Chinese', 'Vietnamese', 'Japanese', 'French', 'Greek', 'Dutch', 'Korean', 'Polish', 'Portuguese', 'Russian', 'Czech', 'German']
国家名种类数:  18


In [8]:
#3、定义函数，实现：读取源数据到内存
def read_data(file_path):
    #1、
    my_list_x,my_list_y = [],[]
    with open(file_path,'r',encoding = 'utf-8') as f:
        for line in f.readlines():
            if len(line) <= 5:
                continue
            x,y = line.strip().split('\t')     #\t 水平制表符
            my_list_x.append(x)
            my_list_y.append(y)
    print(f'my_list_x:{len(my_list_x)}')
    print(f'my_list_y:{len(my_list_y)}')
    return my_list_x,my_list_y

In [9]:
# 创建数据集对象，即：原始数据 -> 数据集对象 TensorDataset -> 数据加载器
class NameClassDataset(Dataset):
    #1、初始化函数
    def __init__(self,my_list_x,my_list_y):
        self.my_list_x = my_list_x
        self.my_list_y = my_list_y
        self.sample_len = len(my_list_x)
    # 定义函数，用于获取样本总数
    def __len__(self):
        return self.sample_len

    #定义函数，实现根据特定索引，获取其对应的样本
    def __getitem__(self,index):
        # 索引边界校验，确保索引在合法范围
        index = min(max(index,0),self.sample_len - 1)
        #按照索引获取原始样本和标签
        x = self.my_list_x[index]
        y = self.my_list_y[index]

        #人名数据转换为 one_hot编码
        tensor_x = torch.zeros(len(x),n_letters)
        for li,letter in enumerate(x):
            letter_index = all_letters.find(letter)
            tensor_x[li][letter_index] = 1
        tensor_y = torch.tensor(categories.index(y),dtype = torch.long)
        return tensor_x,tensor_y
        

In [25]:
def get_dataloader():
    my_list_x,my_list_y = read_data('name_classfication.txt')
    #创建数据集对象
    name_class_dataset = NameClassDataset(my_list_x,my_list_y)

    #参1：
    my_dataloader = DataLoader(name_class_dataset,batch_size = 1,shuffle=True)
    for x,y in my_dataloader:
        print(f'x.shape:{x.shape},x:{x}')
        print(f'y.shape:{y.shape},y:{y}')
        break
    return my_dataloader


In [26]:
class My_RNN(nn.Module):
    #1、初始化函数：输入特征维度，隐藏层维度，输出维度，层数
    def __init__(self,input_size,hidden_size,output_size,n_layers = 1):
        # 1.1 初始化父类成员.
        super().__init__()
        # 1.2 输出特征维度(对应字母表大小, 即: 57个字符)
        self.input_size = input_size
        # 1.3 隐藏层维度, 决定模型的表示能力.
        self.hidden_size = hidden_size
        # 1.4 输出维度(对应国家名数量, 即: 18个国家名)
        self.output_size = output_size
        # 1.5 层数, 默认为1.
        self.n_layers = n_layers
        #1.6 定义RNN
        self.rnn = nn.RNN(self.input_size,self.hidden_size,self.n_layers)
        #1.7 全连接层
        self.linear = nn.Linear(self.hidden_size,self.output_size)
        # 1.8 定义激活函数, 将输出类别 -> 类别的概率分布.
        # 大白话解释: 多分类交叉熵损失函数CrossEntropyLoss(新版写法) = NLLLoss损失函数 + LogSoftmax(dim=-1)  旧版写法
        self.softmax = nn.LogSoftmax(dim=-1)        # 优化2: 如果用CrossEntropyLoss损失函数, 这行代码可以省略不写.

    def forward(self,input,hidden):
        #调整输入张量，添加batch_size
        input = input.unsqueeze(1)

        #通过RNN计算
        #output：所有时间步的隐藏状态，hidden：最后一个时间步的隐藏状态
        output,hn = self.rnn(input,hidden)
        tmp_output = output[-1]
        tmp_output = self.linear(tmp_output)
        return self.softmax(tmp_output),hn

        # 3. 初始化隐藏状态, 创建全0的初始化隐藏状态.
    def init_hidden(self):
        # 参1: 隐藏层层数, 参2: 批次大小, 参3: 隐藏层维度.
        return torch.zeros(self.n_layers, 1, self.hidden_size)



In [18]:
#测试RNN网络模型
def dm_test_myrnn():
    # 1. 实例化RNN对象
    my_rnn = My_RNN(57, 128, 18)
    # print(f'my_rnn: {my_rnn}')      # my_rnn: My_RNN( (rnn): RNN(57, 128) (linear): Linear(in_features=128, out_features=18, bias=True) (softmax): LogSoftmax(dim=-1) )

    # 2. 准备测试数据, 创建1个随机张量, 模拟输入, 形状为: [seq_len人名长度, input_size词向量维度]
    input = torch.randn(6, 57)      # liru: ouyang 欧阳
    print(f'input(输入的张量维度): {input.shape}')     # torch.Size([6, 57])

    # 3. 初始化隐藏状态
    # h0 = torch.zeros(1, 1, 128)
    h0 = my_rnn.init_hidden()       # 效果同上.

    # 4. 测试一次性输入完整的一个样本(序列数据)
    output, hn = my_rnn(input, h0)   #input经unsqueeze之后 [6,1,57], h0[1,1,128]

    # 5. 打印结果.
    print(f'输出的形状: {output.shape}, 输出的内容: {output}')         # [1, 18]
    print(f'隐藏状态的形状: {hn.shape}, 隐藏状态的内容: {hn}')          #  [1, 1, 128]



In [19]:
dm_test_myrnn()

input(输入的张量维度): torch.Size([6, 57])
输出的形状: torch.Size([1, 18]), 输出的内容: tensor([[-2.8122, -3.2417, -2.4324, -2.9224, -3.0584, -3.2259, -2.8717, -3.4114,
         -2.6925, -2.8982, -3.0320, -3.0779, -2.7533, -2.9874, -2.9208, -2.6010,
         -2.5800, -3.0402]], grad_fn=<LogSoftmaxBackward0>)
隐藏状态的形状: torch.Size([1, 1, 128]), 隐藏状态的内容: tensor([[[ 0.0762, -0.3978,  0.1000, -0.5894, -0.0546, -0.2216,  0.3723,
           0.6369,  0.2090,  0.7158,  0.1992, -0.0308, -0.1898, -0.1675,
          -0.5144, -0.4894, -0.0123,  0.3302, -0.5490, -0.0621,  0.3233,
           0.4016,  0.6219, -0.4045, -0.6446,  0.5418, -0.4298,  0.7576,
           0.4680,  0.5441,  0.4769, -0.2071,  0.3508,  0.5077,  0.2304,
           0.1781, -0.3667, -0.1533, -0.1326,  0.4401, -0.0191,  0.5447,
          -0.2421,  0.2934, -0.6342, -0.1432,  0.3345,  0.5644,  0.3043,
          -0.5463,  0.3814, -0.0227,  0.2035,  0.0912,  0.4158, -0.5587,
          -0.1389,  0.3111, -0.1166,  0.2503,  0.5957,  0.0739,  0.4222,
       

In [27]:
# todo 8. 模型训练.
# 定义变量, 记录: 学习率, 训练的轮数.
my_lr, epochs = 1e-3, 1

# todo 8.1 RNN模型训练.
def train_rnn():
    # 1. 数据准备动作.
    # 1.1 读取数据
    my_list_x, my_list_y = read_data('name_classfication.txt')
    # 1.2 构建数据集对象.
    name_class_dataset = NameClassDataset(my_list_x, my_list_y)

    # 2. 模型与优化器初始化.
    # 2.1 定义模型参数,
    # 参1: 输入维度(字符表大小), 参2: 隐藏层维度, 参3: 输出维度(国家数量)
    input_size, n_hidden, output_size = n_letters, 128, category_num       # 等价于: 57, 128, 18

    # 2.2 创建模型对象.
    my_rnn = My_RNN(input_size, n_hidden, output_size)

    # 2.3 定义损失函数和优化器.
    criterion = nn.NLLLoss()    # 如果你用了CrossEntropyLoss(), 则它 = NLLLoss() + LogSoftmax()
    optimizer = optim.Adam(my_rnn.parameters(), lr=my_lr)

    # 3. 训练过程 -> 参数初始化
    start_time = time.time()        # 模型开始训练时间.
    total_iter_num = 0              # 已训练的样本数.
    total_loss = 0.0                # 已训练的损失和
    total_loss_list = []            # 每100个样本求一次平均损失, 形成: 损失列表.
    total_acc_num = 0               # 已训练的样本, 预测准确总数
    total_acc_list = []             # 每100个样本求一次平均准确率, 形成: 准确率列表.

    # 4. 具体的训练过程, 按轮数遍历数据集.
    for epoch in range(epochs):     # epoch: 第几轮
        print(f'\n开始第{epoch + 1}/{epochs} 轮训练...')
        # 4.1 创建数据集加载器对象, 随机打乱数据集.
        train_dataloader = DataLoader(name_class_dataset, batch_size=1, shuffle=True)
        # 4.2 样本迭代训练,  即: 本轮具体的每批次训练
        for i, (x, y) in enumerate(train_dataloader):     # 优化点3: 这里加入进度条.
            # 4.3 前向传播, 计算结果.
            #在 Dataset 里,你的 __getitem__ 返回的 tensor_x 形状是 [seq_len, 57]（例如名字 "Ada"，形状就是 [3, 57]）。
            #在 DataLoader 里,当你设置 batch_size=1 时，DataLoader 会认为你需要以“批次”为单位处理数据。即使只有 1 个样本，它也会在最前面强行加一个维度，表示 Batch Size。
            #所以，DataLoader 吐出来的 x 形状变成了：[1, 3, 57]。
            # x 的形状是 [1, seq_len, 57]
            # x[0] 取出第 0 个样本，形状变回 [seq_len, 57],以便在后续通过unsqueeze ,变成[seq_len,batch_size,57]
            output, hidden = my_rnn(x[0], my_rnn.init_hidden())
            # 4.4 计算损失.
            my_loss = criterion(output, y)
            # 4.5 三剑客 -> 梯度清零, 反向传播, 优化器更新参数.
            optimizer.zero_grad()
            my_loss.backward()
            optimizer.step()

            # 4.6 统计训练结果(指标统计)
            total_iter_num += 1             # 训训练的样本数 + 1
            total_loss += my_loss.item()    # 累计损失值

            # 4.7 计算当前样本预测准确率
            pred_tag = torch.argmax(output).item()
            total_acc_num += (1 if pred_tag == y else 0)        # 统计: 预测正确的样本数

            # 4.8 统计: 每100个样本求一次平均损失, 准确率 形成: 损失列表, 准确率列表.
            if total_iter_num % 100 == 0:
                # 走这里, 说明100步了, 计算: 平均损失.
                avg_loss = total_loss / total_iter_num      # 总损失 / 总样本数
                # 把上述的平均损失, 添加到: 损失列表.
                total_loss_list.append(avg_loss)

                # 计算准确率, 即: 预测正确的 / 总样本数, 并添加到: 准确率列表.
                avg_acc = total_acc_num / total_iter_num
                total_acc_list.append(avg_acc)

            # 4.9 每2000步(个样本), 打印训练日志.
            if total_iter_num % 2000 == 0:
                # 计算平均损失.
                avg_loss = total_loss / total_iter_num
                # 计算模型训练耗时
                end_time = int(time.time() - start_time)
                # 输出训练日志.
                print(f'轮次: {epoch + 1}, 训练的样本数: {total_iter_num}, 平均损失: {avg_loss:.4f}, 耗时: {end_time}s, 准确率: {avg_acc:.4f}')

        # 4.10 走到这里, 说明一轮训练完毕 -> 保存模型.
        torch.save(my_rnn.state_dict(), f'my_rnn_wh02_{epoch + 1}.bin')

    # 5. 走到这里, 训练结束, 返回统计结果.
    total_time = int(time.time() - start_time)
    print(f'训练完成, 总耗时: {total_time}s, 总训练了 {total_iter_num}个样本!!')

    # 6. 优化4: 你可以把下述返回的三个值(损失列表, 训练总耗时, 准确率列表), 存储到文件中.
    #          因为一会儿我们会 可视化3个模型的训练结果, 如果没有存储的话, 会把 训练动作从新跑一次.


    # 7. 返回结果: 损失列表, 训练总耗时, 准确率列表.
    return total_loss_list, total_time, total_acc_list

In [31]:
my_rnn_path = 'my_rnn_wh02_1.bin'

# todo 10.1.2 定义函数, 将要预测的人名 转成 one-hot编码, 例如: 'zhang' -> [5, 57]
def lineToTensor(line):
    # 1. 初始化张量, [文本长度, 字符表长度]
    tensor_x = torch.zeros(len(line), n_letters)

    # 2. 遍历文本, 获取到每个字符及其索引.
    for i, letter in enumerate(line):
        # 3. 查看字符在全局字母表中的位置(索引)
        letter_index = all_letters.find(letter)
        # 4. 在张量的对应位置改为1, 完成: one-hot编码
        tensor_x[i][letter_index] = 1

    # 5. 返回结果.
    return tensor_x         # 即: 'zhang' -> [5, 57]


# todo 10.1.3 定义函数, 实现: RNN预测.
def dm_predict_rnn(x):
    # 1. 定义遍历, 记录模型相关参数.
    n_letters, n_hidden, n_categories = 57, 128, 18
    # 2. 把输入的文字转成 one-hot编码.
    x_tensor = lineToTensor(x)
    # 3. 创建模型对象.
    my_rnn = My_RNN(n_letters, n_hidden, n_categories)
    # 4. 加载模型参数.
    my_rnn.load_state_dict(torch.load(my_rnn_path))
    # 5. 进行预测, 不计算梯度.  -> 节省内存和计算机资源.
    with torch.no_grad():
        # 5.1 模型预测
        output, hidden = my_rnn(x_tensor, my_rnn.init_hidden())
        # 5.2 从预测结果中, 获取前3个最大的元素.
        # 参1(k):   取前3个最大的元素.
        # 参2(dim): 获取概率最大的元素所在的维度.
        # 参3(largest): 获取概率最大的元素.
        topv, topi = output.topk(3, 1, True)
        # 5.3 打印待预测文本.
        print(f'rnn(待预测文本): {x}')

        # 5.4 解析预测结果.
        for i in range(3):
            value = topv[0][i].item()           # 概率值 -> Python的标量
            category_idx  = topi[0][i].item()   # 类别索引
            category = categories[category_idx] # 类别名称.
            print(f'value: {value}, category: {category}')

In [36]:
# 1. 读取数据
my_list_x, my_list_y = read_data('name_classfication.txt')

# 2. 测试: 数据加载器.
get_dataloader()

# 3. 测试RNN模型
dm_test_myrnn()

# 5. 测试: 模型训练.
train_rnn()         # 训练完成, 总耗时: 49s, 总训练了 20074个样本!!

# 7. 测试: 模型预测.
dm_predict_rnn('Piao')

my_list_x:20074
my_list_y:20074
my_list_x:20074
my_list_y:20074
x.shape:torch.Size([1, 8, 57]),x:tensor([[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
          0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 

In [35]:
dm_predict_rnn('Satoshi')

rnn(待预测文本): Satoshi
value: -0.6200658679008484, category: Japanese
value: -1.7213187217712402, category: Italian
value: -2.3996829986572266, category: Arabic
